<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_09-1_RPG_Tool_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 1 | AI RPG 게임 마스터: Tool Calling으로 전투 행동 실행

**실행 환경:** Google Colab (기본 과정은 무료·API 키 불필요)

> **오늘의 핵심:** 모델의 도구 호출 요청(JSON)은 *실행 자체가 아닙니다.* **호스트(Python 코드)**가 함수 이름·인자를 확인하고 함수를 실행하며 결과를 모델에 돌려줍니다.

### 학습 목표
1. 함수 이름·설명·입력 스키마(JSON Schema)를 보고 Tool Calling의 구조를 설명한다.
2. 도구 호출 요청과 Python 함수 실행을 구분한다.
3. 잘못된 인자와 업무 규칙 위반(포션 소진·게임 종료)을 구분해 처리한다.
4. 정상·오류·반복 호출을 자동 검사한다.

**실습 구성:** ① 게임 엔진 → ② 도구 선언 → ③ 모의 Tool Calling → ④ 검증·테스트 → ⑤ (선택) 실제 LLM 연결

*교육용 규칙:* 플레이어 HP 100 / 슬라임 HP 60 / 공격력 20 / 포션 2개(30 회복) / 최대 10턴. 난수 없는 결정적 전투로 학습 결과를 재현합니다.

## 0. 실행 안내

1. Colab에서 **런타임 → 모두 실행**하면 기본 실습이 API 키 없이 끝납니다.
2. 게임 상태가 혼란스러우면 `reset_game()`을 호출하세요.
3. 마지막의 **선택 실습**은 기본값 `RUN_REAL_LLM = False`이므로 자동 실행하지 않습니다. 실제 LLM 사용 시 OpenAI API 키와 이용 요금이 필요합니다.

**중요한 구분:** 아래 모의 응답은 사람이 준비한 JSON이며 **LLM이 생성한 결과가 아닙니다.** 선택 실습에서는 실제 모델의 함수 호출 요청을 받습니다.

In [1]:
# 표준 라이브러리만 사용하므로 별도 설치가 필요하지 않습니다.
from dataclasses import dataclass, asdict
from typing import Any
import json

print("준비 완료: Python 표준 라이브러리만 사용합니다.")

준비 완료: Python 표준 라이브러리만 사용합니다.


## 1. 게임 상태와 규칙 구현

게임 상태는 **호스트 코드가 보유**합니다. 모델은 상태 변경을 직접 수행하지 않고, 변경을 요청하는 도구 호출만 만듭니다.

- `attack_enemy(target)`: 슬라임 HP를 20 감소시킵니다.
- `heal_player()`: 포션이 있으면 HP를 최대 100까지 30 회복합니다.
- `defend()`: 이번 턴 슬라임 공격의 피해를 10에서 5로 낮춥니다.
- `get_game_status()`: 현재 상태만 읽습니다(턴 소모 없음).
- 각 유효한 행동 후 몬스터가 살아 있으면 반격합니다. **게임 종료 후에는 변경이 금지**됩니다.

In [2]:
@dataclass
class GameState:
    # 현재 게임의 원본 상태: 모델 프롬프트가 아니라 Python 메모리에 저장됩니다.
    player_hp: int = 100
    enemy_hp: int = 60
    potions: int = 2
    turn: int = 0
    status: str = "playing"  # playing / win / lose / timeout

# 전투 규칙은 한 곳에 모아 필요할 때 바꾸기 쉽게 만듭니다.
PLAYER_MAX_HP = 100
PLAYER_DAMAGE = 20
ENEMY_DAMAGE = 10
POTION_HEAL = 30
MAX_TURNS = 10

game = GameState()
call_log: list[dict[str, Any]] = []

def reset_game() -> dict[str, Any]:
    """게임 상태와 호출 기록을 모두 초기화합니다."""
    global game
    game = GameState()
    call_log.clear()
    return asdict(game)

def _error(code: str, message: str, **extra: Any) -> dict[str, Any]:
    """모델이 다음 행동을 결정하기 쉽도록 오류를 구조화합니다."""
    return {"ok": False, "error_code": code, "message": message, **extra}

def _enemy_counterattack(defending: bool = False) -> None:
    """유효한 행동 후 살아 있는 슬라임이 1회 반격합니다."""
    if game.enemy_hp > 0:
        damage = ENEMY_DAMAGE // 2 if defending else ENEMY_DAMAGE
        game.player_hp = max(0, game.player_hp - damage)

def _finish_turn() -> None:
    """한 턴 종료 뒤 승리/패배/턴 제한을 판정합니다."""
    game.turn += 1
    if game.enemy_hp <= 0:
        game.status = "win"
    elif game.player_hp <= 0:
        game.status = "lose"
    elif game.turn >= MAX_TURNS:
        game.status = "timeout"

def _snapshot(action: str) -> dict[str, Any]:
    """성공 응답에는 항상 최신 게임 상태를 포함합니다."""
    return {"ok": True, "action": action, "state": asdict(game)}

print(reset_game())

{'player_hp': 100, 'enemy_hp': 60, 'potions': 2, 'turn': 0, 'status': 'playing'}


### 1-1. 실제로 실행할 게임 함수

다음 함수들은 **도구 실행 측(Host)** 코드입니다. 프롬프트나 JSON 자체로는 실행되지 않습니다.

In [3]:
def get_game_status() -> dict[str, Any]:
    """플레이어·몬스터 HP와 포션, 턴 수를 조회합니다. 상태는 변경하지 않습니다."""
    return _snapshot("get_game_status")

def attack_enemy(target: str) -> dict[str, Any]:
    """target='slime'일 때만 공격합니다. 유효한 공격은 턴을 소모합니다."""
    if game.status != "playing":
        return _error("GAME_OVER", "종료된 게임에서는 공격할 수 없습니다.")
    if target != "slime":
        return _error("UNKNOWN_TARGET", "공격 대상이 없습니다.", allowed_targets=["slime"])

    game.enemy_hp = max(0, game.enemy_hp - PLAYER_DAMAGE)
    _enemy_counterattack()  # 슬라임이 죽었다면 이 함수는 아무 일도 하지 않습니다.
    _finish_turn()
    return _snapshot("attack_enemy")

def heal_player() -> dict[str, Any]:
    """포션 1개를 쓰고 체력을 회복합니다. 포션이 없으면 턴을 소모하지 않습니다."""
    if game.status != "playing":
        return _error("GAME_OVER", "종료된 게임에서는 회복할 수 없습니다.")
    if game.potions == 0:
        return _error("NO_POTION", "포션이 없습니다. 공격이나 방어를 선택하세요.", available_actions=["attack_enemy", "defend", "get_game_status"])
    if game.player_hp == PLAYER_MAX_HP:
        return _error("HP_FULL", "HP가 최대치이므로 포션을 소모하지 않습니다.")

    game.potions -= 1
    game.player_hp = min(PLAYER_MAX_HP, game.player_hp + POTION_HEAL)
    _enemy_counterattack()
    _finish_turn()
    return _snapshot("heal_player")

def defend() -> dict[str, Any]:
    """방어 1턴 동안 몬스터 반격 피해를 절반으로 줄입니다."""
    if game.status != "playing":
        return _error("GAME_OVER", "종료된 게임에서는 방어할 수 없습니다.")
    _enemy_counterattack(defending=True)
    _finish_turn()
    return _snapshot("defend")

# 사람이 함수를 직접 호출하면 게임 상태가 즉시 변경됩니다.
reset_game()
print("상태 조회:", get_game_status())
print("공격 실행:", attack_enemy("slime"))

상태 조회: {'ok': True, 'action': 'get_game_status', 'state': {'player_hp': 100, 'enemy_hp': 60, 'potions': 2, 'turn': 0, 'status': 'playing'}}
공격 실행: {'ok': True, 'action': 'attack_enemy', 'state': {'player_hp': 90, 'enemy_hp': 40, 'potions': 2, 'turn': 1, 'status': 'playing'}}


## 2. 모델에 보여줄 도구 정의(JSON Schema)

모델에게 전달되는 것은 **함수 이름, 설명, 파라미터 스키마**입니다. 내부 Python 구현이나 게임 변수 자체가 모델에 자동으로 전달되지는 않습니다.

`strict: true`는 지원되는 JSON Schema 형식을 준수하도록 돕지만 **포션 수·권한·종료 여부 같은 업무 규칙 검증을 대체하지 않습니다.**

In [4]:
# Responses API의 function tool 정의 형태를 사용합니다.
# 모든 객체 스키마는 additionalProperties=False로 제한합니다.
# 인자가 없는 함수는 빈 properties와 required=[]를 사용합니다.
def no_args_schema() -> dict[str, Any]:
    return {"type": "object", "properties": {}, "required": [], "additionalProperties": False}

TOOLS = [
    {
        "type": "function", "name": "attack_enemy",
        "description": "슬라임을 공격합니다. 대상은 slime만 허용하며, 공격에 성공하면 한 턴 진행합니다.",
        "parameters": {
            "type": "object", "properties": {"target": {"type": "string", "enum": ["slime"], "description": "공격 대상"}},
            "required": ["target"], "additionalProperties": False,
        }, "strict": True,
    },
    {
        "type": "function", "name": "heal_player",
        "description": "포션으로 플레이어 HP를 회복합니다. 포션과 HP 상태를 Python 코드가 검증합니다.",
        "parameters": no_args_schema(), "strict": True,
    },
    {
        "type": "function", "name": "defend",
        "description": "한 턴 방어해 몬스터 반격 피해를 줄입니다.",
        "parameters": no_args_schema(), "strict": True,
    },
    {
        "type": "function", "name": "get_game_status",
        "description": "현재 게임 상태(HP, 포션, 턴, 종료 상태)를 조회합니다. 턴을 소모하지 않습니다.",
        "parameters": no_args_schema(), "strict": True,
    },
]

print(json.dumps(TOOLS, ensure_ascii=False, indent=2))

[
  {
    "type": "function",
    "name": "attack_enemy",
    "description": "슬라임을 공격합니다. 대상은 slime만 허용하며, 공격에 성공하면 한 턴 진행합니다.",
    "parameters": {
      "type": "object",
      "properties": {
        "target": {
          "type": "string",
          "enum": [
            "slime"
          ],
          "description": "공격 대상"
        }
      },
      "required": [
        "target"
      ],
      "additionalProperties": false
    },
    "strict": true
  },
  {
    "type": "function",
    "name": "heal_player",
    "description": "포션으로 플레이어 HP를 회복합니다. 포션과 HP 상태를 Python 코드가 검증합니다.",
    "parameters": {
      "type": "object",
      "properties": {},
      "required": [],
      "additionalProperties": false
    },
    "strict": true
  },
  {
    "type": "function",
    "name": "defend",
    "description": "한 턴 방어해 몬스터 반격 피해를 줄입니다.",
    "parameters": {
      "type": "object",
      "properties": {},
      "required": [],
      "additionalProperties": false
    },
    "strict": true
  },
 

## 3. Host의 검증 및 도구 실행기 만들기

중요: JSON의 `name`을 보고 **미리 허용한 함수만** 실행합니다. `eval()`이나 임의의 Python 코드 실행은 사용하지 않습니다.

실제 모델이 만든 인자도 사용자 입력처럼 검사해야 합니다. 여기서는 표준 라이브러리만으로 학습에 필요한 타입·필수값·허용값 검사를 구현합니다.

In [5]:
# 실행 가능한 함수는 명시적으로 허용 목록에 등록합니다.
ALLOWED_FUNCTIONS = {
    "attack_enemy": attack_enemy,
    "heal_player": heal_player,
    "defend": defend,
    "get_game_status": get_game_status,
}
TOOL_DEFINITIONS = {tool["name"]: tool for tool in TOOLS}

def validate_call(name: str, args: Any) -> dict[str, Any] | None:
    """오류가 없으면 None, 문제가 있으면 구조화된 오류를 반환합니다."""
    if name not in ALLOWED_FUNCTIONS:
        return _error("UNKNOWN_TOOL", "허용하지 않는 도구입니다.", allowed_tools=list(ALLOWED_FUNCTIONS))
    if not isinstance(args, dict):
        return _error("INVALID_ARGUMENTS", "arguments는 JSON 객체여야 합니다.")

    schema = TOOL_DEFINITIONS[name]["parameters"]
    required = set(schema["required"])
    provided = set(args)
    if required - provided:
        return _error("MISSING_ARGUMENT", "필수 인자가 빠졌습니다.", missing=sorted(required - provided))
    if provided - set(schema["properties"]):
        return _error("EXTRA_ARGUMENT", "정의되지 않은 인자를 전달했습니다.", extra=sorted(provided - set(schema["properties"])))
    for key, value in args.items():
        prop = schema["properties"][key]
        if prop["type"] == "string" and not isinstance(value, str):
            return _error("INVALID_TYPE", f"{key}에는 문자열이 필요합니다.")
        if "enum" in prop and value not in prop["enum"]:
            return _error("INVALID_VALUE", f"{key}에 허용되지 않은 값입니다.", allowed_values=prop["enum"])
    return None

def execute_tool_call(name: str, arguments: str | dict[str, Any]) -> dict[str, Any]:
    """모델의 요청을 검증한 뒤 허용된 함수만 실행합니다."""
    try:
        args = json.loads(arguments) if isinstance(arguments, str) else arguments
    except json.JSONDecodeError:
        result = _error("INVALID_JSON", "arguments를 JSON으로 해석할 수 없습니다.")
    else:
        problem = validate_call(name, args)
        if problem is not None:
            result = problem
        else:
            # 실제 실행 지점: 모델이 아니라 내 Python 함수가 실행됩니다.
            try:
                result = ALLOWED_FUNCTIONS[name](**args)
            except Exception:
                # 교육용: 내부 예외의 세부 정보는 모델에 그대로 노출하지 않습니다.
                result = _error("INTERNAL_ERROR", "도구 실행 중 내부 오류가 발생했습니다.")
    call_log.append({"tool": name, "arguments": arguments, "result": result})
    return result

reset_game()
print(execute_tool_call("attack_enemy", '{"target":"slime"}'))
print("호출 로그:", call_log[-1])

{'ok': True, 'action': 'attack_enemy', 'state': {'player_hp': 90, 'enemy_hp': 40, 'potions': 2, 'turn': 1, 'status': 'playing'}}
호출 로그: {'tool': 'attack_enemy', 'arguments': '{"target":"slime"}', 'result': {'ok': True, 'action': 'attack_enemy', 'state': {'player_hp': 90, 'enemy_hp': 40, 'potions': 2, 'turn': 1, 'status': 'playing'}}}


## 4. 모의 Tool Calling: 요청 → 검증 → 실행 → 결과

모의 호출을 활용하면 **호출 요청과 실제 실행이 다른 단계**라는 사실을 API 키 없이 재현할 수 있습니다. 아래 `mock_model_output`은 모형 데이터로서, 모델의 선택 능력을 평가하는 결과가 아닙니다.

In [6]:
reset_game()
mock_model_output = {
    "type": "function_call",      # 실제 Responses API에서도 쓰이는 출력 항목 유형
    "call_id": "call_demo_01",     # 호출 결과와 요청을 연결하는 식별자
    "name": "attack_enemy",        # 모델이 요청했다고 가정한 도구 이름
    "arguments": '{"target":"slime"}',  # 인자는 JSON 문자열
}

print("1) 요청만 생성됨. HP는 아직 변하지 않았습니다:", game.enemy_hp)
print("2) 요청 JSON:", mock_model_output)
result = execute_tool_call(mock_model_output["name"], mock_model_output["arguments"])
print("3) Host 실행 결과:", json.dumps(result, ensure_ascii=False))
print("4) 실행 후 슬라임 HP:", game.enemy_hp)
assert game.enemy_hp == 40

1) 요청만 생성됨. HP는 아직 변하지 않았습니다: 60
2) 요청 JSON: {'type': 'function_call', 'call_id': 'call_demo_01', 'name': 'attack_enemy', 'arguments': '{"target":"slime"}'}
3) Host 실행 결과: {"ok": true, "action": "attack_enemy", "state": {"player_hp": 90, "enemy_hp": 40, "potions": 2, "turn": 1, "status": "playing"}}
4) 실행 후 슬라임 HP: 40


### 4-1. 여러 번 호출하는 경우

`get_game_status`는 읽기 전용이며, `attack_enemy`는 상태를 변경합니다. 반복 공격 후 몬스터가 쓰러지면 **추가 공격이 거부**됩니다.

In [7]:
reset_game()
scenario = [
    ("get_game_status", {}),
    ("attack_enemy", {"target": "slime"}),
    ("attack_enemy", {"target": "slime"}),
    ("attack_enemy", {"target": "slime"}),
    ("attack_enemy", {"target": "slime"}),  # 이미 이겼다면 GAME_OVER
]
for name, args in scenario:
    result = execute_tool_call(name, args)
    print(f"{name:16s} → {result.get('error_code', 'OK'):10s} | 상태: {asdict(game)}")
assert game.status == "win"
assert call_log[-1]["result"]["error_code"] == "GAME_OVER"

get_game_status  → OK         | 상태: {'player_hp': 100, 'enemy_hp': 60, 'potions': 2, 'turn': 0, 'status': 'playing'}
attack_enemy     → OK         | 상태: {'player_hp': 90, 'enemy_hp': 40, 'potions': 2, 'turn': 1, 'status': 'playing'}
attack_enemy     → OK         | 상태: {'player_hp': 80, 'enemy_hp': 20, 'potions': 2, 'turn': 2, 'status': 'playing'}
attack_enemy     → OK         | 상태: {'player_hp': 80, 'enemy_hp': 0, 'potions': 2, 'turn': 3, 'status': 'win'}
attack_enemy     → GAME_OVER  | 상태: {'player_hp': 80, 'enemy_hp': 0, 'potions': 2, 'turn': 3, 'status': 'win'}


## 5. 오류 테스트: 스키마 오류 vs 업무 규칙 오류

- **스키마 오류:** 필수 인자 누락, 잘못된 타입, 미허용 도구 → 함수 실행 전에 검증기에서 거부합니다.
- **업무 규칙 오류:** 포션 없음, 이미 종료된 게임 → 실행 함수가 현재 상태를 보고 거부합니다.
- `strict: true`를 쓰더라도 **업무 규칙 검증은 반드시 코드에서** 수행합니다.

In [8]:
reset_game()
invalid_cases = [
    ("attack_enemy", {}),                          # 필수 인자 누락
    ("attack_enemy", {"target": 123}),             # 타입 오류
    ("attack_enemy", {"target": "dragon"}),       # 허용값 위반
    ("attack_enemy", {"target": "slime", "hp": 0}), # 정의하지 않은 인자
    ("delete_all_files", {}),                       # 허용 목록에 없는 도구
    ("attack_enemy", '{broken-json'),               # JSON 형식 오류
]
for name, args in invalid_cases:
    response = execute_tool_call(name, args)
    print(f"{name:16s} → {response['error_code']}: {response['message']}")
    assert response["ok"] is False

# 잘못된 호출은 게임 상태를 바꾸지 않아야 합니다.
assert asdict(game) == asdict(GameState())

attack_enemy     → MISSING_ARGUMENT: 필수 인자가 빠졌습니다.
attack_enemy     → INVALID_TYPE: target에는 문자열이 필요합니다.
attack_enemy     → INVALID_VALUE: target에 허용되지 않은 값입니다.
attack_enemy     → EXTRA_ARGUMENT: 정의되지 않은 인자를 전달했습니다.
delete_all_files → UNKNOWN_TOOL: 허용하지 않는 도구입니다.
attack_enemy     → INVALID_JSON: arguments를 JSON으로 해석할 수 없습니다.


In [9]:
reset_game()
# 포션은 2개지만 처음 HP 100은 최대치이므로 바로 회복할 수 없습니다.
print("HP가 가득 찬 경우:", execute_tool_call("heal_player", {}))
assert game.potions == 2

# 공격 후 반격을 받아 HP가 감소하면 포션을 사용할 수 있습니다.
execute_tool_call("attack_enemy", {"target": "slime"})
print("회복 1:", execute_tool_call("heal_player", {}))
execute_tool_call("attack_enemy", {"target": "slime"})
print("회복 2:", execute_tool_call("heal_player", {}))
print("포션을 모두 쓴 후:", execute_tool_call("heal_player", {}))
assert game.potions == 0
assert call_log[-1]["result"]["error_code"] == "NO_POTION"

# 오류 응답에는 '다음에 선택할 수 있는 행동'도 포함됩니다.
print("선택 가능한 행동:", call_log[-1]["result"]["available_actions"])

HP가 가득 찬 경우: {'ok': False, 'error_code': 'HP_FULL', 'message': 'HP가 최대치이므로 포션을 소모하지 않습니다.'}
회복 1: {'ok': True, 'action': 'heal_player', 'state': {'player_hp': 90, 'enemy_hp': 40, 'potions': 1, 'turn': 2, 'status': 'playing'}}
회복 2: {'ok': True, 'action': 'heal_player', 'state': {'player_hp': 90, 'enemy_hp': 20, 'potions': 0, 'turn': 4, 'status': 'playing'}}
포션을 모두 쓴 후: {'ok': False, 'error_code': 'NO_POTION', 'message': '포션이 없습니다. 공격이나 방어를 선택하세요.', 'available_actions': ['attack_enemy', 'defend', 'get_game_status']}
선택 가능한 행동: ['attack_enemy', 'defend', 'get_game_status']


## 6. 자동 검증 (Contract Test)

아래 테스트는 초깃값, 상태 조회의 읽기 전용 특성, 공격 동작, 잘못된 인자 거부, 포션 오류, 게임 종료 후 거부를 확인합니다. **여기까지 모두 실행되면 기본 실습을 완료한 것입니다.**

In [10]:
def run_contract_tests() -> None:
    # 1) 초기 상태가 정확한지 확인
    reset_game()
    assert (game.player_hp, game.enemy_hp, game.potions, game.turn) == (100, 60, 2, 0)

    # 2) 조회 도구는 턴을 소모하지 않아야 함
    execute_tool_call("get_game_status", {})
    assert game.turn == 0

    # 3) 유효한 공격은 HP를 변경하고 턴을 진행함
    attack_result = execute_tool_call("attack_enemy", {"target": "slime"})
    assert attack_result["ok"] is True and game.enemy_hp == 40 and game.turn == 1

    # 4) 잘못된 스키마는 상태를 바꾸지 않음
    previous = asdict(game).copy()
    invalid = execute_tool_call("attack_enemy", {"target": 123})
    assert invalid["error_code"] == "INVALID_TYPE" and asdict(game) == previous

    # 5) 포션 소진 시 NO_POTION 반환
    reset_game()
    game.player_hp = 40  # 테스트용 상태 준비 (실제 게임 동작이 아님)
    execute_tool_call("heal_player", {})
    execute_tool_call("heal_player", {})
    no_potion = execute_tool_call("heal_player", {})
    assert no_potion["error_code"] == "NO_POTION" and game.potions == 0

    # 6) 종료된 게임은 HP가 더 바뀌지 않음
    reset_game()
    for _ in range(3):
        execute_tool_call("attack_enemy", {"target": "slime"})
    assert game.status == "win"
    previous = asdict(game).copy()
    after_win = execute_tool_call("attack_enemy", {"target": "slime"})
    assert after_win["error_code"] == "GAME_OVER" and asdict(game) == previous

    print("✅ Contract Test 6개 모두 통과")

run_contract_tests()

✅ Contract Test 6개 모두 통과


## 7. (선택) 실제 LLM을 연결해 Tool Calling 확인하기

이전 단계까지는 사람이 준비한 JSON으로 실행 흐름을 학습했습니다. 이제 **실제 모델이 네 가지 도구 중 무엇을 호출할지 선택**하도록 해 봅니다.

- 별도의 OpenAI API 키가 필요하며 **호출 비용이 발생할 수 있습니다.**
- 키는 코드에 직접 저장하지 않고 실행할 때만 입력합니다.
- 실수로 무한 호출하지 않도록 **호출 횟수를 4회로 제한**합니다.
- 새 전투가 시작되도록 상태를 초기화합니다.
- 각 함수 호출은 모델이 아니라 `execute_tool_call()`에서 실제 실행됩니다.

참고: [OpenAI 공식 Function Calling 문서](https://developers.openai.com/api/docs/guides/function-calling)

In [12]:
# 다음 코드는 선택 실행입니다. 전체 실행 시 외부 API를 호출하지 않습니다.
RUN_REAL_LLM = False
MODEL_NAME = "gpt-5.4-mini"  # 사용 가능한 모델로 바꾸어도 됩니다. 계정별 접근 권한과 요금 확인.

if not RUN_REAL_LLM:
    print("선택 실습은 건너뜁니다. 실제 실행하려면 RUN_REAL_LLM=True로 바꾸세요.")
else:
    # Colab 환경에 OpenAI SDK 설치 (선택 실습에서만 수행)
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "-U", "openai"])

    from getpass import getpass
    from openai import OpenAI
    api_key = getpass("OpenAI API 키(입력 내용 숨김): ")
    if not api_key:
        raise ValueError("실제 LLM 실습에는 API 키가 필요합니다.")
    llm_client = OpenAI(api_key=api_key)
    reset_game()

    # 도구 호출을 포함해 상태를 요약해 답하게 유도합니다.
    prompt = "현재 게임 상태를 확인한 뒤 슬라임을 한 번 공격하고 결과를 한국어로 설명해줘."
    response = llm_client.responses.create(
        model=MODEL_NAME,
        instructions="너는 텍스트 RPG 도우미다. 상태는 도구 결과로만 확인한다. 필요한 도구만 호출하라.",
        input=prompt,
        tools=TOOLS,
        parallel_tool_calls=False,  # 교육용으로 한 번에 최대 한 도구 요청
    )

    # 모델은 호출 요청을 반환할 뿐, 게임 함수를 직접 실행하지 못합니다.
    for step in range(4):
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            print("모델 최종 답변:", response.output_text)
            break
        outputs = []
        for call in calls:
            print("모델 호출 요청:", call.name, call.arguments)
            tool_result = execute_tool_call(call.name, call.arguments)
            print("호스트 실행 결과:", tool_result)
            # call_id가 모델이 만든 호출과 실행 결과를 연결해 줍니다.
            outputs.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(tool_result, ensure_ascii=False),
            })
        response = llm_client.responses.create(
            model=MODEL_NAME,
            previous_response_id=response.id,
            input=outputs,
            tools=TOOLS,
            parallel_tool_calls=False,
        )
    else:
        print("안전을 위해 도구 호출 반복을 4단계에서 중단했습니다.")

선택 실습은 건너뜁니다. 실제 실행하려면 RUN_REAL_LLM=True로 바꾸세요.


## 8. 학습 정리

- **Tool 정의:** 함수 이름·설명·입력 스키마를 모델에 알려줍니다.
- **Tool 요청:** 모델이 함수 이름과 인자(JSON)를 제안합니다. 이때 게임 상태는 아직 바뀌지 않습니다.
- **검증·실행:** 호스트 코드가 스키마·허용 목록·게임 규칙을 확인하고 Python 함수를 실행합니다.
- **결과 반환:** 실행 결과를 다시 모델에 전달하면 모델이 이어서 설명하거나 다른 도구를 요청할 수 있습니다.
- **오류 설계:** `NO_POTION`처럼 오류 종류와 다음 가능한 행동을 알려주면 복구하기 쉽습니다.

**다음 실습(2단계):** 동일한 RPG 함수를 MCP Server의 도구로 공개하고 In-memory Client로 호출합니다.

> 주의: 이번 노트북은 실습용 단순 스키마 검증기를 구현했습니다. 실서비스에서는 완전한 JSON Schema 검증, 권한·승인, 재시도·멱등성 등 별도 정책이 필요합니다.